# Lesson 01 — Why the integrator matters

**Phase 1 of the gravsim project: Newton, two bodies.**

Before we can claim to *measure* General Relativity in the solar system we have to be sure
that what we measure is physics and not the numerical method. This lesson builds that
confidence on the simplest problem we have, the Sun and Mercury, where the exact answer is
known (Kepler) and every error is the integrator's fault.

By the end you should be able to

1. write down the equations of motion of the two-body problem and say what an *integrator*
   does with them;
2. explain the difference between a Runge–Kutta scheme and a *symplectic* scheme from the
   behaviour of the energy error;
3. read the *order* of a method off a log–log plot of error against step size;
4. explain why a fixed-step integrator produces a *spurious perihelion precession*, and why
   that matters for a 43 arcsecond-per-century signal.

> **How to use this notebook.** Every section states the physics first and only then shows
> the code. Run the cells in order (Kernel → Restart & Run All takes about ten seconds).
> Exercises are at the end of each section; the ones marked ★ are harder.

## 1. The physics: two bodies, one conic

Two point masses with gravitational parameters $GM_1$ and $GM_2$ (we never use $G$ and the
masses separately; only the products $GM$ are known accurately) attract each other with
Newton's law. Writing $\mathbf r_i$ for the barycentric positions, the equations of motion are

$$
\ddot{\mathbf r}_1 = -\,GM_2\,\frac{\mathbf r_1-\mathbf r_2}{|\mathbf r_1-\mathbf r_2|^3},
\qquad
\ddot{\mathbf r}_2 = -\,GM_1\,\frac{\mathbf r_2-\mathbf r_1}{|\mathbf r_2-\mathbf r_1|^3}.
$$

The *relative* vector $\mathbf r = \mathbf r_2 - \mathbf r_1$ obeys the one-body equation
$\ddot{\mathbf r} = -\mu\,\mathbf r/r^3$ with $\mu = GM_1 + GM_2$, whose bound solutions are
ellipses with the focus at the origin (Kepler's first law). Two scalars are conserved exactly:

* the specific energy $\;\varepsilon = \tfrac12 v^2 - \mu/r = -\mu/(2a)$,
* the specific angular momentum $\;\mathbf h = \mathbf r\times\mathbf v$ (so the orbit is planar and
  sweeps equal areas in equal times).

A third conserved *vector*, the eccentricity (Laplace–Runge–Lenz) vector
$\mathbf e = \mathbf v\times\mathbf h/\mu - \mathbf r/r$, points at the perihelion. Because it is
constant, **a Kepler orbit does not precess**. Any precession we see later is either a real
perturbation (planets, GR) or an integrator artefact. Keeping those two apart is the whole game.

**Units.** Throughout gravsim lengths are in AU, times in days and masses as $GM$ in
AU³/day². In these units $GM_\odot = k^2 \approx 2.959\times10^{-4}$ with Gauss's constant
$k = 0.01720209895$, and the speed of light is $c = 173.14$ AU/day (it will matter in Lesson 04).

In [ ]:
import math

import matplotlib.pyplot as plt
import numpy as np

from gravsim import constants as C
from gravsim.analysis import (
    angular_momentum,
    fit_secular_rate,
    perihelion_longitude_series,
    propagate,
    relative_energy_error,
    state_to_elements,
)
from gravsim.data.horizons import load_j2000
from gravsim.forces import accel_newton
from gravsim.integrators import INTEGRATORS, integrate, integrate_scipy
from gravsim.viz import figure_grid, plot_energy_error, plot_orbits_3d, plot_precession

plt.rcParams["figure.dpi"] = 100
print(f"GM_sun = {C.GM_SUN:.6e} AU^3/day^2,  c = {C.C_AU_DAY:.4f} AU/day")

### Initial conditions: Mercury at J2000.0

gravsim ships the JPL Horizons state vectors of the Sun and the eight planets at
J2000.0 (2000 January 1, 12:00 TDB) in the barycentric J2000-ecliptic frame. We keep only
the Sun and Mercury and move them to *their own* barycentre, so that the pair is an isolated
two-body system at rest.

In [ ]:
state = load_j2000().subset(("Sun", "Mercury")).to_barycentric()
r_rel, v_rel = state.relative_to("Sun")
i_merc = state.index("Mercury")
mu = float(state.gm.sum())  # GM_sun + GM_mercury
el = state_to_elements(r_rel[i_merc], v_rel[i_merc], mu)
period = float(el["period"])
print(f"a = {float(el['a']):.5f} AU,  e = {float(el['e']):.5f},  P = {period:.3f} d,  "
      f"i = {math.degrees(float(el['inc'])):.2f} deg (ecliptic)")
print("barycentre offset after to_barycentric():", np.linalg.norm(state.barycenter()[0]))

## 2. What an integrator does

The computer cannot follow the continuous flow of the ODE; it takes *steps*. Writing the
state as $(\mathbf x, \mathbf v)$ and the force law as $\mathbf a(\mathbf x,\mathbf v,t)$, a
one-step method is a map

$$ (\mathbf x_n, \mathbf v_n) \;\longmapsto\; (\mathbf x_{n+1}, \mathbf v_{n+1}) \qquad t_{n+1}=t_n+\Delta t .$$

A method has **order $p$** when the error committed in one step is $O(\Delta t^{\,p+1})$, so that
after a fixed time span (that is, $\propto 1/\Delta t$ steps) the accumulated error is $O(\Delta t^{\,p})$.
Halving the step divides the error by $2^p$.

gravsim implements the classic ladder (`gravsim.integrators.fixed_step`):

| name | order | one step | symplectic? |
|---|---|---|---|
| `euler` | 1 | $\mathbf x' = \mathbf x + \Delta t\,\mathbf v,\;\; \mathbf v' = \mathbf v + \Delta t\,\mathbf a(\mathbf x)$ (both from the *old* state) | no |
| `symplectic_euler` | 1 | kick $\mathbf v' = \mathbf v + \Delta t\,\mathbf a(\mathbf x)$, then drift with the *new* velocity | yes |
| `leapfrog` | 2 | kick $\tfrac{\Delta t}{2}$ · drift $\Delta t$ · kick $\tfrac{\Delta t}{2}$ (velocity Verlet) | yes |
| `rk4` | 4 | classical Runge–Kutta, four force evaluations | no |
| `yoshida4` | 4 | three leapfrogs with weights $w_1, w_0, w_1$, $w_1 = 1/(2-2^{1/3})$, $w_0 = -2^{1/3} w_1$ | yes |
| `yoshida6` | 6 | seven leapfrogs (Yoshida 1990, solution A) | yes |

"Symplectic" means the map preserves the geometric structure of Hamiltonian flow (phase-space
volume and the symplectic form). The practical consequence, which we will see directly, is that
a symplectic integrator solves *exactly* a slightly perturbed Hamiltonian, so its energy error
**oscillates but never grows**, while a non-symplectic method of the same order lets the energy
**drift**. Notice also the cost column you cannot see: leapfrog needs one force evaluation per
step, RK4 four, Yoshida4 four, Yoshida6 eight.

Every scheme is driven by the same function, `integrate(state, accel, t_end, dt, name)`, which
returns a `Trajectory` with the samples, and which shrinks `dt` minimally so that an integer
number of steps lands exactly on `t_end`.

In [ ]:
names = ["euler", "symplectic_euler", "leapfrog", "rk4", "yoshida4"]
dt = 1.0  # days: 88 steps per orbit
n_orbits = 10
t_end = state.t + n_orbits * period
runs = {name: integrate(state, accel_newton, t_end, dt, name) for name in names}
for name, tr in runs.items():
    print(f"{name:17s} order {INTEGRATORS[name].order}: {tr.meta['n_steps']} steps of {tr.meta['dt']:.4f} d")

### 2.1 The orbits in 3-D

Mercury's orbit is inclined 7° to the ecliptic, so we draw it in three dimensions. The
explicit Euler orbit spirals *outwards*: at every step it uses the velocity of the *beginning* of
the step to move, and the force of the beginning to accelerate, so it always overshoots the
curve on the outside. Symplectic Euler, with the same order, stays bounded.

In [ ]:
fig, axes = figure_grid(2, projection="3d", panel=(5.0, 4.6))
plot_orbits_3d(runs["euler"], ax=axes[0])
axes[0].set_title(f"Explicit Euler, dt = {dt} d, {n_orbits} orbits")
plot_orbits_3d(runs["symplectic_euler"], ax=axes[1])
axes[1].set_title("Symplectic Euler, same step")
for ax in axes:
    ax.view_init(elev=35, azim=-60)

### 2.2 Energy and angular momentum along the run

The conserved quantities are our first error meter. `relative_energy_error` returns
$(E(t)-E_0)/|E_0|$; a perfect integrator would give zero.

* **Euler** gains energy monotonically (the orbit grows).
* **RK4** loses energy monotonically, but at $O(\Delta t^4)$ per orbit, so slowly.
* the **symplectic** schemes oscillate within a band: leapfrog's band is $O(\Delta t^2)$,
  Yoshida4's $O(\Delta t^4)$, and neither drifts.

In [ ]:
fig, axes = figure_grid(2, panel=(5.5, 3.8))
for name, tr in runs.items():
    plot_energy_error(tr.t, relative_energy_error(tr), label=name, ax=axes[0], t_units="days")
axes[0].set_title("|ΔE/E₀| (log scale)")
for name in ("leapfrog", "rk4", "yoshida4"):
    tr = runs[name]
    plot_energy_error(tr.t, relative_energy_error(tr), label=name, ax=axes[1], logy=False, t_units="days")
axes[1].set_title("signed ΔE/E₀: bounded oscillation vs drift")

In [ ]:
# angular momentum: every kick-drift-kick scheme conserves it to roundoff for a central force,
# RK4 and Euler do not.
for name, tr in runs.items():
    L = angular_momentum(tr.pos, tr.vel, tr.gm)
    dL = np.abs(np.linalg.norm(L, axis=1) / np.linalg.norm(L[0]) - 1.0).max()
    print(f"{name:17s} max |ΔL/L₀| = {dL:.2e}")

## 3. Reading the order off a convergence plot

Now the quantitative test. We integrate two orbits with each method and a sequence of step
sizes, and compare the final position with the **exact** Kepler solution
(`gravsim.analysis.propagate`, which solves Kepler's equation). On log–log axes the error of an
order-$p$ method is a straight line of slope $p$.

Watch two things: (i) the slopes, and (ii) the floor around $10^{-14}$–$10^{-12}$ AU where
double-precision roundoff stops any further gain (Yoshida6 hits it first).

In [ ]:
def final_position_error(name, dt, n_orbits=2.0):
    t_end = state.t + n_orbits * period
    tr = integrate(state, accel_newton, t_end, dt, name)
    r_num, _ = tr.relative("Mercury", "Sun")
    r_exact, _ = propagate(r_rel[i_merc], v_rel[i_merc], mu, t_end - state.t)
    return float(np.linalg.norm(r_num[-1] - r_exact))

steps = np.array([8.0, 4.0, 2.0, 1.0, 0.5, 0.25, 0.125])
fig, axes = figure_grid(1, panel=(6.5, 4.5))
ax = axes[0]
for name in ["euler", "symplectic_euler", "leapfrog", "rk4", "yoshida4", "yoshida6"]:
    err = np.array([final_position_error(name, h) for h in steps])
    ax.loglog(steps, err, "o-", label=f"{name} (order {INTEGRATORS[name].order})", ms=4)
for p in (1, 2, 4, 6):
    ax.loglog(steps, 1e-3 * (steps / 1.0) ** p, ":", color="0.6", lw=0.8)
ax.set_xlabel("step size dt [days]")
ax.set_ylabel("|r_num - r_Kepler| after 2 orbits [AU]")
ax.set_title("Convergence: dotted guides have slopes 1, 2, 4, 6")
ax.legend(fontsize=8)
ax.grid(True, which="both", lw=0.3)

> **Check your reading.** Between $\Delta t = 1$ d and $0.5$ d the leapfrog error should fall by
> about 4, RK4 and Yoshida4 by about 16. Verify it in the printed table below. The slope of the
> fitted line is an estimate of the order.

In [ ]:
fine = slice(2, 6)  # dt = 2, 1, 0.5, 0.25 d: inside the asymptotic regime, above roundoff
for name in ["leapfrog", "rk4", "yoshida4"]:
    err = np.array([final_position_error(name, h) for h in steps[fine]])
    slope = np.polyfit(np.log(steps[fine]), np.log(err), 1)[0]
    print(f"{name:10s} error ratio dt=1 -> 0.5: {err[1]/err[2]:6.1f}   fitted order: {slope:.2f}")

## 4. Long runs: the energy error over ten years

Over a decade (about 41 Mercury orbits) the difference between "drift" and "bounded" becomes
the dominant feature. At $\Delta t = 0.5$ d RK4's energy error grows linearly; leapfrog's and
Yoshida4's stay in their bands. Which one would you trust for a *century*?

In [ ]:
t_end = state.t + 10.0 * C.JULIAN_YEAR_DAYS
fig, axes = figure_grid(1, panel=(7.0, 4.0))
long_runs = {}
for name in ("leapfrog", "rk4", "yoshida4", "yoshida6"):
    tr = integrate(state, accel_newton, t_end, 0.5, name, save_every=4)
    long_runs[name] = tr
    plot_energy_error(tr.t, relative_energy_error(tr), label=f"{name}, dt = 0.5 d",
                      ax=axes[0], t_units="years")
axes[0].set_title("Sun + Mercury, 10 years")

## 5. The hidden error: spurious precession

Energy conservation is necessary but not sufficient. A symplectic integrator conserves a
*modified* energy, and the modified problem is **not** exactly a Kepler problem: its orbit is an
ellipse that slowly turns. For leapfrog the turning rate is $O((2\pi/N)^2)$ per orbit, where
$N$ is the number of steps per orbit; for Yoshida4 it is $O((2\pi/N)^4)$.

This is the number that matters for the rest of the course. The GR signal we are after is
**43 arcseconds per century** ($5\times10^{-7}$ rad per orbit). The measurement toolkit of Lesson 03
will give us $\varpi(t)$, the longitude of perihelion; here we only use it to read the slope.

In [ ]:
def spurious_rate(tr):
    t, varpi = perihelion_longitude_series(tr, "Mercury")
    fit = fit_secular_rate(t, varpi)
    return float(fit.rate_arcsec_per_century), fit

rates = {}
for name, tr in long_runs.items():
    rate, _ = spurious_rate(tr)
    rates[name] = rate
    steps_per_orbit = period / tr.meta["dt"]
    print(f"{name:10s} dt = {tr.meta['dt']:.2f} d ({steps_per_orbit:.0f} steps/orbit): "
          f"spurious precession {rate:+12.1f} arcsec/century")
print(f"\nthe GR signal we want to measure: {C.MERCURY_PRECESSION_GR_ARCSEC_CY:+12.2f} arcsec/century")

In [ ]:
fig, axes = figure_grid(1, panel=(7.0, 4.0))
for name in ("leapfrog", "yoshida4", "yoshida6"):
    t, varpi = perihelion_longitude_series(long_runs[name], "Mercury")
    plot_precession(t, varpi, fit_secular_rate(t, varpi).rate, ax=axes[0], label=name)
axes[0].set_title("Longitude of perihelion in a Kepler problem: anything non-zero is the integrator")

How small must the step be for leapfrog to be useful? Its spurious rate scales as $\Delta t^2$, so
reaching $\ll 43''$/cy from $-2\times10^{5}$ at $0.5$ d needs
$\Delta t \lesssim 0.5\,\mathrm d\times\sqrt{1/10^{4}} \approx 7$ minutes — $10^4$ steps per orbit,
$4\times10^6$ steps per century. Possible, but wasteful. Yoshida4's $\Delta t^4$ scaling needs
$\Delta t \approx 0.5\,\mathrm d\,(10^{-2})^{1/4} \approx 2.5$ hours. Verify the scaling yourself:

In [ ]:
for dt_test in (0.5, 0.25):
    tr = integrate(state, accel_newton, state.t + 10 * C.JULIAN_YEAR_DAYS, dt_test, "yoshida4", save_every=int(2 / dt_test))
    print(f"yoshida4 dt = {dt_test} d: {spurious_rate(tr)[0]:+9.1f} arcsec/cy")
print("ratio should be about 2^4 = 16")

## 6. The production integrator: DOP853

For the GR runs gravsim uses SciPy's **DOP853**, an adaptive 8th-order Runge–Kutta pair
(`integrate_scipy`, tolerances `rtol=1e-12, atol=1e-15`). It is not symplectic — its energy
error drifts — but at this tolerance the drift is $\sim10^{-13}$ per century, and its spurious
precession is about $0.001''$/century: four orders of magnitude below the GR signal. It also
handles velocity-dependent forces (the 1PN term) without any special treatment, which the
kick-drift-kick schemes do not.

In [ ]:
t_eval = state.t + np.arange(0.0, 10 * C.JULIAN_YEAR_DAYS + 1.0, 1.0)
dop = integrate_scipy(state, accel_newton, t_eval)
rate_dop, _ = spurious_rate(dop)
print(f"DOP853: {dop.meta['nfev']} force evaluations, max |ΔE/E| = "
      f"{np.abs(relative_energy_error(dop)).max():.1e}, spurious precession {rate_dop:+.4f} arcsec/cy")
print(f"noise floor / GR signal = {abs(rate_dop) / C.MERCURY_PRECESSION_GR_ARCSEC_CY:.1e}")

## 7. Summary

| question | answer from this lesson |
|---|---|
| Does the method conserve energy? | Symplectic ones: bounded error. RK: drift. Euler: hopeless. |
| How fast does the error shrink with the step? | As $\Delta t^p$; read $p$ off the log–log plot. |
| Is energy conservation enough? | **No.** Symplectic schemes still precess spuriously at $O((2\pi/N)^p)$. |
| What do we use for GR? | DOP853 at `rtol=1e-12`: noise floor $\sim10^{-3}\,''$/cy. |

## Exercises

1. Repeat the energy-error plot of §2.2 with $\Delta t = 0.25$ d. By which factor does each band
   shrink? (Leapfrog: 4; Yoshida4: 16.)
2. The harmonic oscillator $\ddot x = -x$ has period $2\pi$. Write the force law
   `def accel(pos, vel, gm, t): return -pos` and integrate one period with leapfrog at
   $N = 10, 20, 40$ steps. Show that the *phase* error per period is $\propto 1/N^2$.
3. ★ Leapfrog's modified Hamiltonian is $H + \tfrac{\Delta t^2}{12}\{\ldots\}$. Using the
   spurious rates printed in §5, estimate the coefficient of the $(2\pi/N)^2$ law for
   Mercury's eccentricity, and compare with the value you get at $\Delta t = 0.25$ d.
4. ★ Replace Mercury by Neptune (period 165 years). With the same $\Delta t = 0.5$ d the number of
   steps per orbit is 700 times larger. Predict the leapfrog spurious rate before running it.